# Create Manual Labeling Subset and Training Pool

This notebook takes the extracted app review dataset and creates two files:

1. **Manual labeling subset** – a manageable sample you can open in Excel or Google Sheets and label manually.
2. **Training pool / remaining reviews** – all other reviews not selected for manual labeling.

The actual train/test/validation split should be done later during model development, after manual labels exist.


In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 80)


## Configuration

In [ ]:
# Input file
INPUT_XLSX = Path("../data/raw/reviews_final_hek_viactiv.xlsx")

# Fallback if you run this notebook directly in the same folder as the XLSX file
if not INPUT_XLSX.exists():
    raise FileNotFoundError("Input XLSX file not found. Please check the path and ensure the file is in the correct location.")

# Output folder
LABELING_DIR = Path("../data/labeling")
LABELING_DIR.mkdir(parents=True, exist_ok=True)

# Manual labeling sample size
LABEL_SAMPLE_SIZE = 300

# Reproducibility
RANDOM_STATE = 42

# Sample roughly balanced across rating groups and source stores
BALANCE_BY = ["source_store", "rating_group"]


## Load and inspect data

In [ ]:
df = pd.read_excel(INPUT_XLSX)

print(f"Loaded reviews: {len(df)}")
display(df.head())
print("\nColumns:")
print(df.columns.tolist())


## Standardize helper columns

This adds a `rating_group` column if it does not already exist and ensures dates/ratings are usable.


In [ ]:
def rating_group(value):
    try:
        rating = float(value)
    except (TypeError, ValueError):
        return np.nan

    # Collapse the numeric star scale into broader sentiment-oriented groups.
    if rating <= 2:
        return "negative"
    if rating == 3:
        return "neutral"
    if rating >= 4:
        return "positive"
    return np.nan


df = df.copy()

# Standardize review_date
if "review_date" in df.columns:
    df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce", utc=True)
    df["review_date"] = df["review_date"].dt.tz_convert(None)

# Ensure numeric rating
if "rating" in df.columns:
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Add rating_group if missing or empty
if "rating_group" not in df.columns or df["rating_group"].isna().all():
    df["rating_group"] = df["rating"].apply(rating_group)

# Ensure review_id exists
if "review_id" not in df.columns:
    df["review_id"] = [f"review_{i:06d}" for i in range(len(df))]

# Remove rows without text
df = df[df["review_text"].notna()].copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"] != ""].copy()

# Remove exact duplicate review IDs if present
df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

print(f"Reviews after basic cleaning: {len(df)}")
display(df[["review_id", "source_store", "review_date", "rating", "rating_group", "review_text"]].head())


## Check distribution

This helps verify whether the sample should be balanced by store and rating group.


In [ ]:
print("Reviews by source store:")
display(df["source_store"].value_counts(dropna=False).to_frame("count"))

print("\nReviews by rating group:")
display(df["rating_group"].value_counts(dropna=False).to_frame("count"))

print("\nReviews by source store and rating group:")
display(pd.crosstab(df["source_store"], df["rating_group"], margins=True))


## Create manual labeling sample

### Sampling strategy

The review dataset is likely imbalanced across rating groups and stores, so a purely random sample would overrepresent the most common review types. To improve the usefulness of the manual labeling set, this notebook uses a stratified sampling strategy with minimum quotas for selected rating groups. The goal is to create a diverse and manageable annotation subset while keeping the remaining reviews available for later modeling.


In [ ]:
def stratified_sample(
    data: pd.DataFrame,
    sample_size: int,
    strata_cols: list[str],
    random_state: int = 42
) -> pd.DataFrame:
    if sample_size >= len(data):
        return data.copy().sample(frac=1, random_state=random_state)

    if not strata_cols:
        return data.sample(n=sample_size, random_state=random_state)

    if len(strata_cols) == 1:
        strata = data[strata_cols[0]]
    else:
        strata = data[strata_cols].astype(str).agg(" | ".join, axis=1)

    # Estimate how strongly each stratum is represented in the full dataset so the sample can mirror the original distribution as closely as possible.
    proportions = strata.value_counts(normalize=True)
    target_counts = (proportions * sample_size).round().astype(int)

    diff = sample_size - target_counts.sum()
    if diff != 0:
        order = proportions.sort_values(ascending=False).index.tolist()
        i = 0
        while diff != 0 and order:
            key = order[i % len(order)]
            if diff > 0:
                target_counts[key] += 1
                diff -= 1
            else:
                if target_counts[key] > 0:
                    target_counts[key] -= 1
                    diff += 1
            i += 1

    # Sample each stratum separately and combine the results afterward.
    sampled_parts = []
    for key, n in target_counts.items():
        if n <= 0:
            continue
        if len(strata_cols) == 1:
            mask = data[strata_cols[0]] == key
        else:
            mask = data[strata_cols].astype(str).agg(" | ".join, axis=1) == key
        group_df = data[mask]
        if len(group_df) == 0:
            continue
        sampled_parts.append(group_df.sample(n=min(n, len(group_df)), random_state=random_state))

    out = pd.concat(sampled_parts).drop_duplicates()
    if len(out) > sample_size:
        out = out.sample(n=sample_size, random_state=random_state)
    return out.sample(frac=1, random_state=random_state).reset_index(drop=True)

def stratified_sample_with_minimums(
    data: pd.DataFrame,
    sample_size: int,
    rating_col: str = "rating_group",
    store_col: str = "source_store",
    min_per_rating_group: dict | None = None,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Create a sample that guarantees minimum counts for selected rating groups
    and fills the remaining sample proportionally from the remaining data.

    This is useful when the dataset is strongly imbalanced, e.g. mostly negative reviews.
    """

    if sample_size >= len(data):
        return data.copy().sample(frac=1, random_state=random_state).reset_index(drop=True)

    data = data.copy()

    # Define explicit minimum quotas for underrepresented review types so the final labeling sample is not dominated by the largest class only.
    if min_per_rating_group is None:
        min_per_rating_group = {
            "neutral": 20,
            "positive": 50
        }

    # Build the sample in two stages:
    # first satisfy minimum quotas, then fill the remaining slots in a balanced way.
    sampled_parts = []
    used_indices = set()

    # 1. First sample minimum quotas for underrepresented rating groups
    for group, min_n in min_per_rating_group.items():
        group_df = data[data[rating_col] == group]

        n = min(min_n, len(group_df))

        if n > 0:
            sample_part = group_df.sample(n=n, random_state=random_state)
            sampled_parts.append(sample_part)
            used_indices.update(sample_part.index)

        print(f"{group}: sampled {n} of {len(group_df)} available")

    # 2. Fill the remaining slots proportionally from all remaining reviews
    current_n = sum(len(part) for part in sampled_parts)
    remaining_n = sample_size - current_n

    remaining_data = data.drop(index=list(used_indices))

    if remaining_n > 0:
        # If store column exists, keep store distribution roughly balanced in the fill
        strata_cols = [col for col in [store_col, rating_col] if col in remaining_data.columns]

        fill_sample = stratified_sample(
            data=remaining_data,
            sample_size=remaining_n,
            strata_cols=strata_cols,
            random_state=random_state
        )

        sampled_parts.append(fill_sample)

    final_sample = pd.concat(sampled_parts, ignore_index=False)

    # 3. Safety checks
    if len(final_sample) > sample_size:
        final_sample = final_sample.sample(n=sample_size, random_state=random_state)

    final_sample = final_sample.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return final_sample


manual_sample = stratified_sample_with_minimums(
    data=df,
    sample_size=LABEL_SAMPLE_SIZE,
    rating_col="rating_group",
    store_col="source_store",
    min_per_rating_group={
        "neutral": 20,
        "positive": 50
    },
    random_state=42
)

pd.crosstab(
    manual_sample["source_store"],
    manual_sample["rating_group"],
    margins=True
)

# The training pool contains all reviews that were not selected for manual labeling.
training_pool = df[~df["review_id"].isin(manual_sample["review_id"])].copy().reset_index(drop=True)

print(f"Manual labeling sample: {len(manual_sample)}")
print(f"Training pool / remaining reviews: {len(training_pool)}")

print("\nManual sample distribution:")
display(pd.crosstab(manual_sample["source_store"], manual_sample["rating_group"], margins=True))


## Prepare manual labeling file

The labeling file is designed to be easy to use in Excel or Google Sheets.

Columns to be manually filled:

- `label_topics_raw`: free-text topic labels you discover inductively, separated by semicolon, e.g. `login; document upload`
- `label_sentiment`: optional manual text sentiment, e.g. `positive`, `neutral`, `negative`
- `label_relevant`: optional flag, e.g. `yes`, `no`, `unclear`
- `label_notes`: optional comments on difficult cases

No predefined topic labels are included at this stage.


In [ ]:
# Columns to include in the manual labeling file.
base_columns = [
    "review_id",
    "source_store",
    "app_name",
    "app_identifier",
    "review_date",
    "rating",
    "rating_group",
    "review_title",
    "review_text",
    "review_language",
    "review_version",
    "country",
    "thumbs_up_count",
    "developer_response",
]

base_columns = [col for col in base_columns if col in manual_sample.columns]

manual_labeling_df = manual_sample[base_columns].copy()

# Add empty columns for manual labeling
manual_labeling_df["label_topics_raw"] = ""

# Sort for easier labeling: low ratings first, then date
sort_cols = [col for col in ["rating", "review_date"] if col in manual_labeling_df.columns]
manual_labeling_df = manual_labeling_df.sort_values(
    sort_cols,
    ascending=[True, False]
).reset_index(drop=True)

display(manual_labeling_df.head())

## Export files

### Export for manual annotation

The manual labeling file is intended for human review in Excel or Google Sheets. It contains the most relevant review metadata plus empty columns for label entry. The remaining reviews are saved separately so they can be used later as a training pool or for additional sampling rounds.

### Exports:

- `manual_labeling_sample.csv`
- `manual_labeling_sample.xlsx`
- `training_pool_unlabeled.csv`

In [ ]:
manual_xlsx_path = LABELING_DIR / "manual_labeling_pool_hek_viactiv.xlsx"
training_pool_path = LABELING_DIR / "training_pool_hek_viactiv.xlsx"

manual_labeling_df.to_excel(manual_xlsx_path, index=False)

training_pool.to_excel(training_pool_path, index=False)

print("Exported files:")
print(f"- {manual_xlsx_path.resolve()}")
print(f"- {training_pool_path.resolve()}")


## Re-load exported labeling file check

In [ ]:
check = pd.read_excel(manual_xlsx_path)
print(f"Rows in exported Excel labeling file: {len(check)}")
display(check.head())
